<a href="https://colab.research.google.com/github/Joammp/ML_Radio_Signal/blob/main/BUSCA_HP_Classes_1cap(3).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#IMPORTS

In [ ]:
!pip install scikeras
!pip install tensorflow
!pip install kagglehub

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np # Import numpy for potential future use

from matplotlib import pyplot as plt

import h5py
import json
from numpy import argwhere

from sklearn.model_selection import train_test_split
import numpy as np

from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np


import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import ModelCheckpoint
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import GridSearchCV
import threading # Usaremos threading para garantir a segurança da thread
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

import numpy as np
import tensorflow as tf
from tensorflow import keras
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Dropout, Flatten, Dense
from tensorflow.keras.optimizers import Adam, SGD
from sklearn.model_selection import RandomizedSearchCV
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt


In [ ]:
# Before creating the plot, set the desired font size and name
plt.rcParams['font.size'] = 12  # Change the default font size
plt.rcParams['font.family'] = 'serif'  # Change the default font family (e.g., 'serif', 'sans-serif', 'monospace')

# You can also change these settings for specific text elements if needed.
# For example, to change the title font size:
# plt.title('Plot Title', fontsize=16)


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("pinxau1000/radioml2018")

print("Path to dataset files:", path)

h5py_path = path + '/GOLD_XYZ_OSC.0001_1024.hdf5'
modulation_classes_path = path + '/classes-fixed.json'
# Open the dataset
hdf5_file = h5py.File(h5py_path, 'r')
# Load the modulation classes. You can also copy and paste the content of classes-fixed.txt.
modulation_classes = json.load(open(modulation_classes_path, 'r'))

# Read the HDF5 groups
data = hdf5_file['X']
modulation_onehot = hdf5_file['Y']
snr = hdf5_file['Z']

Using Colab cache for faster access to the 'radioml2018' dataset.
Path to dataset files: /kaggle/input/radioml2018


In [ ]:


# Load all data from the datasets in the HDF5 file
X = data
Y = modulation_onehot
Z = snr

# Convert one-hot encoded labels to numerical labels
y = np.argmax(Y, axis=1)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SEED GLOBAL — garante reprodutibilidade total entre execuções
# Cole esta célula como a PRIMEIRA célula de código do notebook
# ══════════════════════════════════════════════════════════════════════════════

import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Garante operações determinísticas na GPU (pequeno custo de performance)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

print(f"✅ Seeds fixadas: SEED={SEED}")
print(f"   random     : OK")
print(f"   numpy      : OK")
print(f"   torch CPU  : OK")
print(f"   torch CUDA : OK")
print(f"   cudnn det. : OK")

✅ Seeds fixadas: SEED=42
   random     : OK
   numpy      : OK
   torch CPU  : OK
   torch CUDA : OK
   cudnn det. : OK


In [ ]:
import torch
import os
from tqdm import tqdm

def train_and_validate(model, train_loader, val_loader, criterion, optimizer,
                       scheduler=None, epochs=10, save_name="best_model"):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # 📁 Diretório padrão do Colab
    SAVE_DIR = "/content"
    os.makedirs(SAVE_DIR, exist_ok=True)

    best_val_acc = 0.0
    best_model_path = None
    epochs_no_improve = 0

    # 🔥 Early stopping baseado no scheduler
    if scheduler is not None and isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
        early_stop_patience = scheduler.patience * 2
    else:
        early_stop_patience = 10

    print(f"🛑 Early Stopping patience: {early_stop_patience}")

    for epoch in range(epochs):

        # ── TREINO ──
        model.train()
        train_loss, train_correct, train_total = 0, 0, 0

        pbar_train = tqdm(train_loader, desc=f"Época {epoch+1}/{epochs} [Treino]")

        for inputs, labels in pbar_train:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = outputs.max(1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        # ── VALIDAÇÃO ──
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)

                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        # ── MÉTRICAS ──
        final_train_loss = train_loss / len(train_loader)
        final_train_acc  = 100. * train_correct / train_total
        final_val_loss   = val_loss / len(val_loader)
        final_val_acc    = 100. * val_correct / val_total

        current_lr = optimizer.param_groups[0]['lr']

        # ── CHECKPOINT MELHOR ──
        if final_val_acc > best_val_acc:
            best_val_acc = final_val_acc
            epochs_no_improve = 0

            # 🧹 remove modelo anterior
            if best_model_path and os.path.exists(best_model_path):
                os.remove(best_model_path)

            # 🏷 nome com métricas
            filename = (
                f"{save_name}_ep{epoch+1:03d}"
                f"_acc{final_val_acc:.2f}"
                f"_lr{current_lr:.2e}.pth"
            )

            best_model_path = os.path.join(SAVE_DIR, filename)

            torch.save({
                'epoch': epoch + 1,
                'val_acc': final_val_acc,
                'val_loss': final_val_loss,
                'lr': current_lr,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
            }, best_model_path)

            print(f"💾 Novo melhor modelo salvo:")
            print(f"   📁 {best_model_path}")

        else:
            epochs_no_improve += 1

        # ── SCHEDULER ──
        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(final_val_acc)  # 🔥 melhor usar ACC
            else:
                scheduler.step()

        # ── LOG ──
        print(f"\nÉpoca {epoch+1}: "
              f"Train Loss: {final_train_loss:.4f} | Train Acc: {final_train_acc:.2f}% | "
              f"Val Loss: {final_val_loss:.4f} | Val Acc: {final_val_acc:.2f}% | "
              f"LR: {current_lr:.2e}")

        # ── EARLY STOP ──
        if epochs_no_improve >= early_stop_patience:
            print(f"\n🛑 Early stopping ativado na época {epoch+1}")
            break

    print(f"\n✅ Melhor Val Acc: {best_val_acc:.2f}%")
    print(f"📁 Modelo salvo em: {best_model_path}")

    return best_model_path

In [ ]:


def get_indices_by_snr(target_snrs):
    """
    Separates and returns indices corresponding to specified SNR values.

    Args:
        target_snrs (list or np.ndarray): A list or array of SNR values
                                          (e.g., [-20, -10, 0, 10]) to filter by.

    Returns:
        np.ndarray: A sorted numpy array of indices from the original dataset (X, Y, Z, y)
                    where the SNR matches one of the target_snrs.
    """
    # Convert target_snrs to a set for efficient lookup
    target_snrs_set = set(target_snrs)

    # Get the SNR values from the Z array (assuming Z is globally available)
    # Z[:, 0] extracts the SNR column
    all_snr_values = Z[:, 0]

    # Find indices where the SNR value is in the target_snrs_set
    matching_indices = np.where(np.isin(all_snr_values, list(target_snrs_set)))[0]

    return np.sort(matching_indices)

# Example Usage:
# Let's say you want indices for SNR values of -20dB, 0dB, and 10dB
# desired_snrs = [-20, 0, 10]
# selected_snr_indices = get_indices_by_snr(desired_snrs)
#
# print(f"Found {len(selected_snr_indices)} samples with SNR in {desired_snrs}")
#
# # You can then use these indices to access the corresponding data
# # X_filtered_by_snr = X[selected_snr_indices]
# # y_filtered_by_snr = y[selected_snr_indices]
# # Z_filtered_by_snr = Z[selected_snr_indices]

# The previous content of this cell was likely a print statement.
# I'll add a simple print to confirm the function is defined.


In [ ]:
def get_samples_by_snr(target_snrs):
    """
    Returns data samples (X, Y, Z, y) corresponding to specified SNR values.

    Args:
        target_snrs (list or np.ndarray): A list or array of SNR values
                                          (e.g., [-20, -10, 0, 10]) to filter by.

    Returns:
        tuple: A tuple containing X_filtered, Y_filtered, Z_filtered, y_filtered
               for the selected SNR values.
    """
    # Use the existing function to get indices for the target SNRs
    selected_indices = get_indices_by_snr(target_snrs)

    # Use these indices to subset the global data arrays
    X_filtered = X[selected_indices]
    Y_filtered = Y[selected_indices]
    Z_filtered = Z[selected_indices]
    y_filtered = y[selected_indices]

    print(f"\nSubset created with {len(selected_indices)} samples for SNR values: {target_snrs}.")
    print(f"Shape of X_filtered: {X_filtered.shape}")
    print(f"Shape of Y_filtered: {Y_filtered.shape}")
    print(f"Shape of Z_filtered: {Z_filtered.shape}")
    print(f"Shape of y_filtered: {y_filtered.shape}")

    return X_filtered, Y_filtered, Z_filtered, y_filtered, selected_indices

print("Function 'get_samples_by_snr' defined.")

Function 'get_samples_by_snr' defined.


In [ ]:
def subset_snr (target_snrs, frac):
    X_snr, Y_snr, Z_snr, y_snn, selected_indices = get_samples_by_snr(target_snrs)

    ind_snr_frac, ind_lixo, Y_snr_frac, Ysnr_lixo = train_test_split(
        selected_indices,
        Y_snr,
        test_size=1-frac,
        random_state=42,
        stratify=Y_snr
    )

    ind_snr_frac = np.sort(ind_snr_frac)
    X_filtered = X[ind_snr_frac]
    Y_filtered = Y[ind_snr_frac]
    Z_filtered = Z[ind_snr_frac]
    y_filtered = y[ind_snr_frac]

    print(f"\nSubset created with {len(selected_indices)} samples for SNR values: {target_snrs}.")
    print(f"Shape of X_filtered: {X_filtered.shape}")
    print(f"Shape of Y_filtered: {Y_filtered.shape}")
    print(f"Shape of Z_filtered: {Z_filtered.shape}")
    print(f"Shape of y_filtered: {y_filtered.shape}")

    return X_filtered, Y_filtered, Z_filtered, y_filtered, ind_snr_frac





In [ ]:
def split (indices, lables):
  train_val_indices, test_indices, train_val_lables, test_lables = train_test_split(
        indices,
        lables,
        test_size=0.2,
        random_state=42,
        stratify=lables
    )

  train_indices, val_indices, train_lables, val_lables = train_test_split(
        train_val_indices,
        train_val_lables,
        test_size=0.25,
        random_state=42,
        stratify=train_val_lables
    )

  train_indices_sorted = np.sort(train_indices)
  val_indices_sorted = np.sort(val_indices)
  test_indices_sorted = np.sort(test_indices)

  return train_indices_sorted, val_indices_sorted, test_indices_sorted

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ACESSO UNIVERSAL AO GOOGLE DRIVE — via OAuth (conta pessoal)
#
# Service Accounts NÃO funcionam com contas Gmail pessoais para gravar
# arquivos (elas não têm cota de armazenamento própria e só escrevem em
# Shared Drives, que exigem Google Workspace pago). Por isso usamos OAuth
# com um token da própria conta jmarcopereira29@gmail.com: uma vez gerado,
# esse token funciona em QUALQUER ambiente (Colab, servidor, Jupyter local),
# sem precisar logar de novo, e sempre acessa o Drive dessa mesma conta.
#
# ── COMO CONFIGURAR (faça isso uma única vez) ──────────────────────────────
# 1. Acesse https://console.cloud.google.com/apis/credentials
#    (crie um projeto, se ainda não tiver um)
# 2. Clique em "+ CRIAR CREDENCIAIS" → "ID do cliente OAuth"
#    - Se pedir para configurar a "Tela de consentimento OAuth" antes, escolha
#      tipo "Externo", preencha nome/e-mail, e em "Test users" adicione
#      jmarcopereira29@gmail.com
#    - Tipo de aplicativo: "App para computador" (Desktop app)
#    - Dê um nome e clique em Criar
# 3. Baixe o JSON gerado e salve como 'client_secret.json'
# 4. No SEU COMPUTADOR (onde tem navegador), rode o script auxiliar
#    'gerar_token_drive.py' (fornecido separadamente). Ele abre o navegador,
#    você loga como jmarcopereira29@gmail.com e autoriza o acesso.
#    Isso gera o arquivo 'token.json'.
# 5. Suba 'token.json' (e 'client_secret.json') para qualquer ambiente onde
#    este notebook for rodar — não precisa repetir o login.
# ══════════════════════════════════════════════════════════════════════════════

!pip install -q google-api-python-client google-auth google-auth-oauthlib

import os
import io
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request

# ▶▶ AJUSTE APENAS ESTA LINHA ◀◀
DRIVE_FOLDER_ID = "1E2bJyP18S4xq4OhBbvgryJ0Oc5Hm_tv2"

SCOPES              = ["https://www.googleapis.com/auth/drive"]
CLIENT_SECRET_FILE  = "client_secret.json"
TOKEN_FILE          = "token.json"

if not os.path.exists(TOKEN_FILE):
    raise FileNotFoundError(
        f"Não encontrei '{TOKEN_FILE}'.\n"
        f"Rode o script 'gerar_token_drive.py' no seu computador (com "
        f"navegador) logado como jmarcopereira29@gmail.com para gerar esse "
        f"arquivo, depois faça upload dele (e do 'client_secret.json') "
        f"aqui neste ambiente."
    )

_drive_creds = Credentials.from_authorized_user_file(TOKEN_FILE, SCOPES)

# Renova o token automaticamente se estiver expirado
if _drive_creds.expired and _drive_creds.refresh_token:
    _drive_creds.refresh(Request())
    with open(TOKEN_FILE, "w") as f:
        f.write(_drive_creds.to_json())

_drive_service = build("drive", "v3", credentials=_drive_creds)

# Confirma qual conta está autenticada
_about = _drive_service.about().get(fields="user").execute()
print(f"✅ Conectado ao Google Drive como: {_about['user']['emailAddress']}")
print(f"   Pasta raiz (Drive Folder ID): {DRIVE_FOLDER_ID}")


# ══════════════════════════════════════════════════════════════════════════════
# FUNÇÕES DE SINCRONIZAÇÃO — usadas em vez de leitura/escrita direta no Drive
# ══════════════════════════════════════════════════════════════════════════════

_drive_folder_cache = {}  # relpath da pasta -> folder_id (evita buscas repetidas)


def _drive_get_or_create_folder(rel_dir):
    """Garante que a árvore de pastas 'rel_dir' existe no Drive e retorna o ID da pasta final."""
    if rel_dir in ("", "."):
        return DRIVE_FOLDER_ID
    if rel_dir in _drive_folder_cache:
        return _drive_folder_cache[rel_dir]

    parent_id = DRIVE_FOLDER_ID
    partial = ""
    for part in rel_dir.replace("\\", "/").split("/"):
        if not part:
            continue
        partial = f"{partial}/{part}" if partial else part
        if partial in _drive_folder_cache:
            parent_id = _drive_folder_cache[partial]
            continue

        query = (
            f"'{parent_id}' in parents and name = '{part}' "
            f"and mimeType = 'application/vnd.google-apps.folder' and trashed = false"
        )
        resp = _drive_service.files().list(q=query, fields="files(id, name)").execute()
        files = resp.get("files", [])

        if files:
            folder_id = files[0]["id"]
        else:
            metadata = {
                "name": part,
                "mimeType": "application/vnd.google-apps.folder",
                "parents": [parent_id],
            }
            folder = _drive_service.files().create(body=metadata, fields="id").execute()
            folder_id = folder["id"]

        _drive_folder_cache[partial] = folder_id
        parent_id = folder_id

    return parent_id


def _drive_find_file(rel_dir, filename):
    """Retorna o file_id de 'filename' dentro de 'rel_dir' no Drive, ou None se não existir."""
    parent_id = _drive_get_or_create_folder(rel_dir)
    query = f"'{parent_id}' in parents and name = '{filename}' and trashed = false"
    resp = _drive_service.files().list(q=query, fields="files(id, name)").execute()
    files = resp.get("files", [])
    return files[0]["id"] if files else None


def drive_push(local_path):
    """
    Envia (cria ou atualiza) um arquivo local para o Drive, espelhando o
    caminho relativo a DRIVE_BASE. Ex.: DRIVE_BASE/PSK/kfold_summary.json
    """
    rel_path = os.path.relpath(local_path, DRIVE_BASE)
    rel_dir  = os.path.dirname(rel_path)
    filename = os.path.basename(rel_path)

    parent_id   = _drive_get_or_create_folder(rel_dir)
    existing_id = _drive_find_file(rel_dir, filename)
    media       = MediaFileUpload(local_path, resumable=True)

    if existing_id:
        _drive_service.files().update(fileId=existing_id, media_body=media).execute()
    else:
        metadata = {"name": filename, "parents": [parent_id]}
        _drive_service.files().create(body=metadata, media_body=media, fields="id").execute()


def drive_pull(local_path):
    """
    Baixa do Drive para 'local_path' (espelhando o caminho relativo a
    DRIVE_BASE), se o arquivo existir remotamente. Retorna True se baixou,
    False se não havia nada no Drive ainda.
    """
    rel_path = os.path.relpath(local_path, DRIVE_BASE)
    rel_dir  = os.path.dirname(rel_path)
    filename = os.path.basename(rel_path)

    file_id = _drive_find_file(rel_dir, filename)
    if file_id is None:
        return False

    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    request = _drive_service.files().get_media(fileId=file_id)
    with io.FileIO(local_path, "wb") as fh:
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            _, done = downloader.next_chunk()
    return True


✅ Conectado ao Google Drive como: jmarcopereira29@gmail.com
   Pasta raiz (Drive Folder ID): 1E2bJyP18S4xq4OhBbvgryJ0Oc5Hm_tv2


In [ ]:
from torch.utils.data import Dataset
class H5PyDataset(Dataset):
    def __init__(self, h5_filepath, data_X_name, data_Y_name, data_Z_name, indices, formats):
        self.h5_filepath = h5_filepath
        self.data_X_name = data_X_name
        self.data_Y_name = data_Y_name
        self.indices = np.array(indices)
        self.formats = formats
        self.file = None

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        # Abre o arquivo apenas uma vez por worker
        if self.file is None:
            self.file = h5py.File(self.h5_filepath, 'r')
            self.X_data = self.file[self.data_X_name]
            self.Y_data = self.file[self.data_Y_name]

        # Pega o índice real baseado na sua lista de sorteados
        real_idx = self.indices[idx]

        # Busca UMA amostra (1024, 2)
        x = self.X_data[real_idx, :]
        y = self.Y_data[real_idx]

        # Normalização Individual
        # mean = x.mean()
        # std = x.std()
        # x = (x - mean) / (std + 1e-8)

        # Trata o Label
        if self.formats == 0:
            # Se y for um vetor (One-Hot), pega o índice da classe
            if hasattr(y, "__len__"):
                y = np.argmax(y)

        # Conversão para Tensor
        x_tensor = torch.from_numpy(x).float() # [1024, 2]
        x_tensor = x_tensor.permute(1, 0)      # [2, 1024] (Canais primeiro)
        y_tensor = torch.tensor(y).long()      # Valor escalar

        return x_tensor, y_tensor

In [ ]:
num_samples = X.shape[0]
indices = np.arange(num_samples) #selected_indices_sorted
lables = np.column_stack((y, Z))

train_indices_sorted, val_indices_sorted, test_indices_sorted = split(indices, lables)

In [ ]:
from torch.utils.data import DataLoader, BatchSampler, RandomSampler, SequentialSampler, random_split

# 1. Dataset (Certifique-se que o __getitem__ aceita a lista de índices como discutimos)
# Dataset de Treino
train_dataset = H5PyDataset(
    h5_filepath=h5py_path,
    data_X_name='X',
    data_Y_name='Y',
    data_Z_name='Z',
    indices=train_indices_sorted,  # Seus índices de treino
    formats=0
)

# Dataset de Validação
val_dataset = H5PyDataset(
    h5_filepath=h5py_path,
    data_X_name='X',
    data_Y_name='Y',
    data_Z_name='Z',
    indices=val_indices_sorted,    # Seus índices de validação
    formats=0
)

test_dataset = H5PyDataset(
    h5_filepath=h5py_path,
    data_X_name='X',
    data_Y_name='Y',
    data_Z_name='Z',
    indices=test_indices_sorted,    # Seus índices de validação
    formats=0
)

batch_size = 64

# Remova o BatchSampler e o collate_fn.
# Use shuffle=True para o treino e shuffle=False para validação.
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,          # Isso substitui o RandomSampler
    drop_last=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,         # Isso substitui o SequentialSampler
    drop_last=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,         # Isso substitui o SequentialSampler
    drop_last=False
)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# DATALOADERS COM SEED CONTROLADA
# ══════════════════════════════════════════════════════════════════════════════

import torch
from torch.utils.data import DataLoader

SEED       = 42
batch_size = 64

# Gerador dedicado para o DataLoader — controla o shuffle entre épocas
g = torch.Generator()
g.manual_seed(SEED)

def seed_worker(worker_id):
    """Garante seeds determinísticas nos workers do DataLoader."""
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=False,
    num_workers=0,
    persistent_workers=False,
    generator=g,               # controla a ordem do shuffle
    worker_init_fn=seed_worker # controla seeds dos workers (se num_workers>0)
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    drop_last=False,
    num_workers=0,
    persistent_workers=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    drop_last=False,
    num_workers=0,
    persistent_workers=False
)

In [ ]:
import torch.nn as nn
class CNN(nn.Module):
    def __init__(self, num_classes):
        super(CNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv1d(2, 64, kernel_size=11, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(64, 128, kernel_size=7, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(128, 256, kernel_size=5, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(512, 1024, kernel_size=3, padding=1),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.MaxPool1d(2)


        )

        # Técnica para detectar o tamanho do flatten automaticamente:
        self.flatten = nn.Flatten()
        with torch.no_grad():
            # Passamos um dado "dummy" de teste para ver o que sai das convs
            dummy_input = torch.zeros(1, 2, 1024)
            dummy_output = self.features(dummy_input)
            self.n_flatten = dummy_output.view(1, -1).size(1)

        print(f"Tamanho detectado para o Flatten: {self.n_flatten}")

        self.classifier = nn.Sequential(
            nn.Linear(self.n_flatten, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.flatten(x)
        x = self.classifier(x)
        return x

In [ ]:
# -*- coding: utf-8 -*-
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║   BUSCA DE ARQUITETURA — K-Fold CV com retomada automática entre grupos     ║
# ║                                                                              ║
# ║  Comportamento:                                                              ║
# ║    • Detecta salvamentos no Drive e continua de onde parou                  ║
# ║    • Ao terminar um grupo segue automaticamente para o próximo              ║
# ║    • LR cai pela metade a cada 2 épocas sem melhora                        ║
# ║    • Early stopping após 10 épocas sem melhora                             ║
# ║    • Divisão de dados invariante entre sessões (seed fixo + salvo no Drive) ║
# ║    • Labels remapeados automaticamente para [0, num_classes-1]              ║
# ║                                                                              ║
# ║  Pré-requisitos (células anteriores já executadas):                         ║
# ║    • kagglehub.dataset_download() → variável path                           ║
# ║    • modulation_classes_path definido                                       ║
# ║    • Classes H5PyDataset, CNN, FlexCNN definidas                            ║
# ║    • Função split() definida                                                 ║
# ║    • Drive montado                                                           ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import os, json, time, copy, random, shutil
import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import StratifiedKFold, train_test_split
from tqdm import tqdm

# ══════════════════════════════════════════════════════════════════════════════
# ▶▶  CONFIGURAÇÃO — altere apenas este bloco  ◀◀
# ══════════════════════════════════════════════════════════════════════════════

# Ordem dos grupos a processar — o script segue esta sequência automaticamente
# Grupos já completos são pulados; retoma no grupo interrompido
GRUPOS_ALVO = ["AM","APSK","QAM"]
# FM tem 1 classe — não precisa de busca de arquitetura

DESIRED_SNRS     = [0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30]

K_FOLDS          = 5
EPOCHS_PER_FOLD  = 120      # máximo de épocas por fold
LR_PATIENCE      = 5       # épocas sem melhora → LR cai pela metade
EARLY_STOP       = 15      # épocas sem melhora → encerra o fold
LR_MIN           = 1e-9    # LR mínimo (não cai abaixo disso)
SEED             = 42
DROPOUT          = 0.5
LEARNING_RATE    = 1e-3
BATCH_SIZE       = 64
CLASSIFIER_HEAD  = [512]

DRIVE_BASE       = "/content/drive_cache/radioml_sessions"  # cache local; sincronizado com o Drive via drive_push/drive_pull
LOCAL_DIR        = "/content"
HDF5_BUILD_BATCH = 2048

ARCHITECTURES = [
    {
        "label": "2L_32-64",
        "arch": [
            {"out_channels": 32,  "kernel_size": 7, "pool": True},
            {"out_channels": 64,  "kernel_size": 5, "pool": True},
        ],
    },
    {
        "label": "2L_64-128",
        "arch": [
            {"out_channels": 64,  "kernel_size": 7, "pool": True},
            {"out_channels": 128, "kernel_size": 5, "pool": True},
        ],
    },
    {
        "label": "3L_32-64-128",
        "arch": [
            {"out_channels": 32,  "kernel_size": 7, "pool": True},
            {"out_channels": 64,  "kernel_size": 5, "pool": True},
            {"out_channels": 128, "kernel_size": 3, "pool": True},
        ],
    },
    {
        "label": "3L_64-128-256",
        "arch": [
            {"out_channels": 64,  "kernel_size": 7, "pool": True},
            {"out_channels": 128, "kernel_size": 5, "pool": True},
            {"out_channels": 256, "kernel_size": 3, "pool": True},
        ],
    },
    {
        "label": "3L_128-256-512",
        "arch": [
            {"out_channels": 128, "kernel_size": 7, "pool": True},
            {"out_channels": 256, "kernel_size": 5, "pool": True},
            {"out_channels": 512, "kernel_size": 3, "pool": True},
        ],
    },
    {
        "label": "4L_32-64-128-256",
        "arch": [
            {"out_channels": 32,  "kernel_size": 11, "pool": True},
            {"out_channels": 64,  "kernel_size": 7,  "pool": True},
            {"out_channels": 128, "kernel_size": 5,  "pool": True},
            {"out_channels": 256, "kernel_size": 3,  "pool": True},
        ],
    },
    {
        "label": "4L_64-128-256-512",
        "arch": [
            {"out_channels": 64,  "kernel_size": 11, "pool": True},
            {"out_channels": 128, "kernel_size": 7,  "pool": True},
            {"out_channels": 256, "kernel_size": 5,  "pool": True},
            {"out_channels": 512, "kernel_size": 3,  "pool": True},
        ],
    },
    {
        "label": "4L_128-256-512-512",
        "arch": [
            {"out_channels": 128, "kernel_size": 11, "pool": True},
            {"out_channels": 256, "kernel_size": 7,  "pool": True},
            {"out_channels": 512, "kernel_size": 5,  "pool": True},
            {"out_channels": 512, "kernel_size": 3,  "pool": True},
        ],
    },
    {
        "label": "5L_32-64-128-256-512",
        "arch": [
            {"out_channels": 32,   "kernel_size": 11, "pool": True},
            {"out_channels": 64,   "kernel_size": 7,  "pool": True},
            {"out_channels": 128,  "kernel_size": 5,  "pool": True},
            {"out_channels": 256,  "kernel_size": 3,  "pool": True},
            {"out_channels": 512,  "kernel_size": 3,  "pool": True},
        ],
    },
    {
        "label": "5L_64-128-256-512-1024",
        "arch": [
            {"out_channels": 64,   "kernel_size": 11, "pool": True},
            {"out_channels": 128,  "kernel_size": 7,  "pool": True},
            {"out_channels": 256,  "kernel_size": 5,  "pool": True},
            {"out_channels": 512,  "kernel_size": 3,  "pool": True},
            {"out_channels": 1024, "kernel_size": 3,  "pool": True},
        ],
    },
    {
        "label": "6L_64-64-128-128-256-512_mixpool",
        "arch": [
            {"out_channels": 64,  "kernel_size": 11, "pool": True},
            {"out_channels": 64,  "kernel_size": 7,  "pool": False},
            {"out_channels": 128, "kernel_size": 5,  "pool": True},
            {"out_channels": 128, "kernel_size": 3,  "pool": False},
            {"out_channels": 256, "kernel_size": 3,  "pool": True},
            {"out_channels": 512, "kernel_size": 3,  "pool": True},
        ],
    },
    {
        "label": "6L_32-64-64-128-256-512_mixpool",
        "arch": [
            {"out_channels": 32,  "kernel_size": 11, "pool": True},
            {"out_channels": 64,  "kernel_size": 7,  "pool": False},
            {"out_channels": 64,  "kernel_size": 5,  "pool": True},
            {"out_channels": 128, "kernel_size": 3,  "pool": False},
            {"out_channels": 256, "kernel_size": 3,  "pool": True},
            {"out_channels": 512, "kernel_size": 3,  "pool": True},
        ],
    },
]

# ══════════════════════════════════════════════════════════════════════════════
# MAPA DE GRUPOS
# ══════════════════════════════════════════════════════════════════════════════

GROUP_MAP = {
    "OOK":       0, "4ASK":      0, "8ASK":      0,
    "BPSK":      1, "QPSK":      1, "8PSK":      1,
    "16PSK":     1, "32PSK":     1, "GMSK":      1, "OQPSK":     1,
    "16APSK":    2, "32APSK":    2, "64APSK":    2, "128APSK":   2,
    "16QAM":     3, "32QAM":     3, "64QAM":     3, "128QAM":    3, "256QAM":    3,
    "AM-SSB-WC": 4, "AM-SSB-SC": 4, "AM-DSB-WC": 4, "AM-DSB-SC": 4,
    "FM":        5,
}
GROUP_NAMES = {0: "ASK", 1: "PSK", 2: "APSK", 3: "QAM", 4: "AM", 5: "FM"}

sep = "═" * 65

# ══════════════════════════════════════════════════════════════════════════════
# SEEDS
# ══════════════════════════════════════════════════════════════════════════════

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

# ══════════════════════════════════════════════════════════════════════════════
# FlexCNN
# ══════════════════════════════════════════════════════════════════════════════

class FlexCNN(nn.Module):
    def __init__(self, num_classes, arch, classifier=None,
                 dropout=0.5, in_channels=2, input_length=1024):
        super().__init__()
        if classifier is None:
            classifier = [512]

        layers = []
        ch_in  = in_channels
        for block in arch:
            ch_out = block["out_channels"]
            ks     = block["kernel_size"]
            layers += [
                nn.Conv1d(ch_in, ch_out, kernel_size=ks, padding=ks // 2),
                nn.BatchNorm1d(ch_out),
                nn.ReLU(inplace=True),
            ]
            if block.get("pool", True):
                layers.append(nn.MaxPool1d(2))
            ch_in = ch_out

        self.features = nn.Sequential(*layers)
        self.flatten  = nn.Flatten()

        with torch.no_grad():
            dummy  = torch.zeros(1, in_channels, input_length)
            n_flat = self.features(dummy).view(1, -1).size(1)

        head = []
        prev = n_flat
        for units in classifier:
            head += [nn.Linear(prev, units), nn.ReLU(inplace=True),
                     nn.Dropout(dropout)]
            prev  = units
        head.append(nn.Linear(prev, num_classes))
        self.classifier = nn.Sequential(*head)

    def forward(self, x):
        return self.classifier(self.flatten(self.features(x)))

# ══════════════════════════════════════════════════════════════════════════════
# FUNÇÕES AUXILIARES
# ══════════════════════════════════════════════════════════════════════════════

def _drive_dir(modelo):
    """Retorna e cria o diretório do modelo no Drive."""
    d = os.path.join(DRIVE_BASE, modelo)
    os.makedirs(d, exist_ok=True)
    return d


def _idx_path(modelo, name):
    return os.path.join(_drive_dir(modelo), f"{name}.npy")


def _h5_local(modelo):
    snr_tag = '_'.join(str(s) for s in DESIRED_SNRS)
    return os.path.join(LOCAL_DIR, f"{modelo}_subset_snr_{snr_tag}.hdf5")


def _h5_drive(modelo):
    snr_tag = '_'.join(str(s) for s in DESIRED_SNRS)
    return os.path.join(_drive_dir(modelo),
                        f"{modelo}_subset_snr_{snr_tag}.hdf5")


def _results_path(modelo):
    return os.path.join(_drive_dir(modelo), "kfold_fold_results.json")


def _summary_path(modelo):
    return os.path.join(_drive_dir(modelo), "kfold_summary.json")


def _folds_path(modelo):
    return os.path.join(_drive_dir(modelo),
                        f"kfold_folds_k{K_FOLDS}.json")


# ══════════════════════════════════════════════════════════════════════════════
# PASSO 1 — HDF5 FILTRADO COM LABELS REMAPEADOS
# ══════════════════════════════════════════════════════════════════════════════

def ensure_hdf5(modelo, mod_classes):
    """
    Garante que o HDF5 filtrado existe localmente com labels remapeados.
    Tenta carregar do Drive; se não existir, constrói do zero.
    Sempre verifica e corrige labels globais → locais.
    """
    h5_loc  = _h5_local(modelo)
    h5_drv  = _h5_drive(modelo)
    snr_set = set(DESIRED_SNRS)

    target_id       = {v: k for k, v in GROUP_NAMES.items()}[modelo]
    group_indices   = np.array(
        sorted([i for i, m in enumerate(mod_classes)
                if GROUP_MAP[m] == target_id]),
        dtype=np.int64
    )
    num_classes     = len(group_indices)
    global_to_local = {int(g): l for l, g in enumerate(group_indices)}

    # ── Carrega do Drive se disponível ───────────────────────────────────────
    if not os.path.exists(h5_loc):
        print(f"  📥 Verificando se '{modelo}' já existe no Drive...")
        if drive_pull(h5_loc):
            print(f"  📥 HDF5 baixado do Drive → {h5_loc}")
        else:
            # Constrói do zero
            print(f"  🔨 Construindo HDF5 para '{modelo}'...")
            _build_hdf5(h5_loc, mod_classes, group_indices,
                        global_to_local, num_classes, snr_set)
            # Persiste no Drive
            drive_push(h5_loc)
            print(f"  💾 HDF5 salvo no Drive: {h5_drv}")

    # ── Verifica/corrige labels ───────────────────────────────────────────────
    _fix_labels(h5_loc, global_to_local, num_classes)

    return h5_loc, num_classes, group_indices, global_to_local


def _build_hdf5(h5_out, mod_classes, group_indices,
                global_to_local, num_classes, snr_set):
    """Filtra INPUT_FILE por grupo e SNR, grava h5_out com labels locais."""
    input_file = path + '/GOLD_XYZ_OSC.0001_1024.hdf5'
    selected   = []

    with h5py.File(input_file, 'r') as src:
        N    = src['X'].shape[0]
        n_b  = int(np.ceil(N / HDF5_BUILD_BATCH))
        for i in range(n_b):
            s, e    = i * HDF5_BUILD_BATCH, min((i+1)*HDF5_BUILD_BATCH, N)
            z_batch = src['Z'][s:e, 0]
            y_batch = np.argmax(src['Y'][s:e], axis=1)
            mask    = (np.isin(z_batch, list(snr_set)) &
                       np.isin(y_batch, group_indices))
            selected.extend((np.where(mask)[0] + s).tolist())
            if (i+1) % 20 == 0 or i == n_b-1:
                print(f"    Batch {i+1}/{n_b} | selecionados: {len(selected)}")

    selected = np.array(selected, dtype=np.int64)
    M        = len(selected)

    with h5py.File(input_file, 'r') as src:
        x_shape = src['X'].shape[1:]
        z_shape = src['Z'].shape[1:]

        with h5py.File(h5_out, 'w') as out:
            out.attrs['modelo_id']      = str(group_indices[0])  # placeholder
            out.attrs['num_classes']    = num_classes
            out.attrs['group_indices']  = group_indices.tolist()
            out.attrs['snrs']           = DESIRED_SNRS
            out.attrs['total_samples']  = M
            out.attrs['labels_remapped']= True

            ds_X = out.create_dataset('X', shape=(M,)+x_shape,
                                      dtype=src['X'].dtype)
            ds_Z = out.create_dataset('Z', shape=(M,)+z_shape,
                                      dtype=src['Z'].dtype)
            ds_Y = out.create_dataset('Y', shape=(M, num_classes),
                                      dtype=np.int32)

            n_b2 = int(np.ceil(M / HDF5_BUILD_BATCH))
            for i in range(n_b2):
                s, e  = i*HDF5_BUILD_BATCH, min((i+1)*HDF5_BUILD_BATCH, M)
                idx   = selected[s:e]
                ds_X[s:e] = src['X'][idx]
                ds_Z[s:e] = src['Z'][idx]

                y_global   = np.argmax(src['Y'][idx], axis=1)
                y_local    = np.array([global_to_local[int(g)]
                                       for g in y_global])
                y_onehot   = np.zeros((len(y_local), num_classes), dtype=np.int32)
                y_onehot[np.arange(len(y_local)), y_local] = 1
                ds_Y[s:e]  = y_onehot

                if (i+1) % 10 == 0 or i == n_b2-1:
                    print(f"    Gravando batch {i+1}/{n_b2}")

    print(f"  ✅ HDF5 construído: {M} amostras, {num_classes} classes")


def _fix_labels(h5_path, global_to_local, num_classes):

    # Abre somente para leitura
    with h5py.File(h5_path, "r") as f:

        if f.attrs.get("labels_remapped", False):
            return

        y_sample = np.argmax(f["Y"][:100], axis=1)

    # <-- o arquivo já foi fechado aqui

    if y_sample.max() < num_classes:
        with h5py.File(h5_path, "a") as f:
            f.attrs["labels_remapped"] = True
        return

    print("Corrigindo labels...")

    with h5py.File(h5_path, "r") as f:
        y_global = np.argmax(f["Y"][:], axis=1)
        N = len(y_global)

    y_local = np.array([global_to_local[int(g)] for g in y_global])

    y_onehot = np.zeros((N, num_classes), dtype=np.int32)
    y_onehot[np.arange(N), y_local] = 1

    with h5py.File(h5_path, "a") as f:
        del f["Y"]
        f.create_dataset("Y", data=y_onehot)
        f.attrs["labels_remapped"] = True
        f.attrs["num_classes"] = num_classes
# ══════════════════════════════════════════════════════════════════════════════
# PASSO 2 — SPLIT DETERMINÍSTICO
# ══════════════════════════════════════════════════════════════════════════════

def ensure_split(modelo, h5_loc):
    """Carrega split do Drive ou gera e salva."""
    tp = _idx_path(modelo, "train_indices")
    vp = _idx_path(modelo, "val_indices")
    ep = _idx_path(modelo, "test_indices")

    # Tenta trazer do Drive antes de checar o cache local
    if not os.path.exists(tp):
        drive_pull(tp)
    if not os.path.exists(vp):
        drive_pull(vp)
    if not os.path.exists(ep):
        drive_pull(ep)

    if os.path.exists(tp) and os.path.exists(vp) and os.path.exists(ep):
        train_idx = np.load(tp)
        val_idx   = np.load(vp)
        test_idx  = np.load(ep)
        print(f"  ✅ Split carregado do Drive  "
              f"(treino={len(train_idx):,}  "
              f"val={len(val_idx):,}  "
              f"teste={len(test_idx):,})")
        return train_idx, val_idx, test_idx

    print(f"  Gerando split (seed={SEED})...")
    with h5py.File(h5_loc, 'r') as f:
        M       = f['X'].shape[0]
        y_lbl   = np.argmax(f['Y'][:], axis=1)

    indices = np.arange(M)
    tv_idx, te_idx, tv_y, _ = train_test_split(
        indices, y_lbl, test_size=0.2,
        random_state=SEED, stratify=y_lbl
    )
    tr_idx, vl_idx, _, _ = train_test_split(
        tv_idx, tv_y, test_size=0.25,
        random_state=SEED, stratify=tv_y
    )

    train_idx = np.sort(tr_idx)
    val_idx   = np.sort(vl_idx)
    test_idx  = np.sort(te_idx)

    np.save(tp, train_idx)
    np.save(vp, val_idx)
    np.save(ep, test_idx)
    drive_push(tp)
    drive_push(vp)
    drive_push(ep)
    print(f"  ✅ Split salvo no Drive  "
          f"(treino={len(train_idx):,}  "
          f"val={len(val_idx):,}  "
          f"teste={len(test_idx):,})")
    return train_idx, val_idx, test_idx

# ══════════════════════════════════════════════════════════════════════════════
# PASSO 3 — FOLDS DETERMINÍSTICOS
# ══════════════════════════════════════════════════════════════════════════════

def ensure_folds(modelo, h5_loc, train_idx):
    """Carrega folds do Drive ou gera e salva."""
    fp = _folds_path(modelo)

    if not os.path.exists(fp):
        drive_pull(fp)

    if os.path.exists(fp):
        with open(fp) as f:
            data = json.load(f)
        assert data["k_folds"]  == K_FOLDS,          "k_folds diverge!"
        assert data["seed"]     == SEED,              "seed diverge!"
        assert data["n_train"]  == len(train_idx),    "tamanho de treino mudou!"
        splits = [(np.array(d["train_idx"]),
                   np.array(d["val_idx"]))
                  for d in data["folds"]]
        print(f"  ✅ Folds carregados do Drive: {fp}")
        return splits

    print(f"  Gerando {K_FOLDS} folds (seed={SEED})...")
    with h5py.File(h5_loc, 'r') as f:
        y_train = np.argmax(f['Y'][train_idx], axis=1)

    skf    = StratifiedKFold(n_splits=K_FOLDS, shuffle=True,
                              random_state=SEED)
    splits = list(skf.split(np.arange(len(train_idx)), y_train))

    data = {
        "modelo": modelo, "k_folds": K_FOLDS,
        "seed": SEED, "n_train": len(train_idx),
        "folds": [
            {"fold": i, "n_train": len(tr), "n_val": len(vl),
             "train_idx": tr.tolist(), "val_idx": vl.tolist()}
            for i, (tr, vl) in enumerate(splits)
        ]
    }
    with open(fp, "w") as f:
        json.dump(data, f)
    drive_push(fp)

    print(f"  ✅ Folds salvos: {fp}")
    for i, (tr, vl) in enumerate(splits):
        print(f"     Fold {i+1}: treino={len(tr):,}  val={len(vl):,}")
    return splits

# ══════════════════════════════════════════════════════════════════════════════
# PASSO 4 — CARREGAR / SALVAR RESULTADOS
# ══════════════════════════════════════════════════════════════════════════════

def load_results(modelo):
    rp = _results_path(modelo)
    if not os.path.exists(rp):
        drive_pull(rp)
    if os.path.exists(rp):
        with open(rp) as f:
            data = json.load(f)
        done = {(r["label"], r["fold"]) for r in data}
        print(f"  ✅ {len(data)} resultados anteriores  "
              f"({len(done)} combinações concluídas)")
        return data, done
    return [], set()


def save_fold_result(modelo, result, all_results):
    all_results.append(result)
    rp = _results_path(modelo)
    with open(rp, "w") as f:
        json.dump(all_results, f, indent=2, ensure_ascii=False)
    drive_push(rp)


def update_summary(modelo, all_results, num_classes):
    from collections import defaultdict
    stats = defaultdict(list)
    meta  = {}
    for r in all_results:
        stats[r["label"]].append(r["best_val_acc"])
        if r["label"] not in meta:
            meta[r["label"]] = {
                k: r[k] for k in ["arch", "n_layers", "filters",
                                   "classifier", "dropout", "lr"]
                if k in r
            }

    summary = []
    for label, accs in stats.items():
        e = {
            "modelo"       : modelo,
            "num_classes"  : num_classes,
            "label"        : label,
            "folds_done"   : len(accs),
            "complete"     : len(accs) == K_FOLDS,
            "accs_per_fold": [round(a, 4) for a in accs],
            "mean_val_acc" : round(float(np.mean(accs)), 4),
            "std_val_acc"  : round(float(np.std(accs)),  4),
            "var_val_acc"  : round(float(np.var(accs)),  4),
            "min_val_acc"  : round(float(np.min(accs)),  4),
            "max_val_acc"  : round(float(np.max(accs)),  4),
        }
        e.update(meta.get(label, {}))
        summary.append(e)

    summary.sort(key=lambda x: (x["complete"], x["mean_val_acc"]),
                 reverse=True)
    sp = _summary_path(modelo)
    with open(sp, "w") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)
    drive_push(sp)
    return summary


def grupo_completo(modelo):
    """Retorna True se todos os folds de todas as arquiteturas estão prontos."""
    rp = _results_path(modelo)
    if not os.path.exists(rp):
        drive_pull(rp)
    if not os.path.exists(rp):
        return False
    with open(rp) as f:
        data = json.load(f)
    done = {(r["label"], r["fold"]) for r in data}
    total = len(ARCHITECTURES) * K_FOLDS
    return len(done) == total

# ══════════════════════════════════════════════════════════════════════════════
# TREINO DE UM FOLD
# ══════════════════════════════════════════════════════════════════════════════

def train_one_fold(model, tr_loader, vl_loader, device,
                   lr, epochs, lr_patience, early_stop, lr_min):
    """
    Treina um fold com:
      • LR cai pela metade a cada `lr_patience` épocas sem melhora
      • Early stop após `early_stop` épocas sem melhora
    Retorna (best_val_acc, history)
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # Scheduler manual — mais granular que ReduceLROnPlateau padrão
    best_val_acc      = 0.0
    best_state        = None
    epochs_no_improve = 0
    history           = []
    model.to(device)

    for epoch in range(epochs):

        # ── Treino ────────────────────────────────────────────────────────────
        model.train()
        tr_loss = tr_correct = tr_total = 0
        pbar = tqdm(tr_loader,
                    desc=f"    Ép {epoch+1:>3}/{epochs} [tr]",
                    leave=False, ncols=80)
        for xb, yb in pbar:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out  = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            tr_loss    += loss.item()
            tr_correct += (out.argmax(1) == yb).sum().item()
            tr_total   += yb.size(0)
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        # ── Validação ─────────────────────────────────────────────────────────
        model.eval()
        vl_loss = vl_correct = vl_total = 0
        with torch.no_grad():
            for xb, yb in vl_loader:
                xb, yb = xb.to(device), yb.to(device)
                out     = model(xb)
                loss    = criterion(out, yb)
                vl_loss    += loss.item()
                vl_correct += (out.argmax(1) == yb).sum().item()
                vl_total   += yb.size(0)

        tr_acc  = 100.0 * tr_correct / tr_total
        vl_acc  = 100.0 * vl_correct / vl_total
        tr_loss /= len(tr_loader)
        vl_loss /= len(vl_loader)
        cur_lr   = optimizer.param_groups[0]["lr"]

        history.append({
            "epoch"     : epoch + 1,
            "train_loss": round(tr_loss, 4),
            "train_acc" : round(tr_acc,  2),
            "val_loss"  : round(vl_loss, 4),
            "val_acc"   : round(vl_acc,  2),
            "lr"        : cur_lr,
        })

        print(f"    Ép {epoch+1:>3}/{epochs} │ "
              f"tr={tr_loss:.4f}/{tr_acc:.2f}% │ "
              f"vl={vl_loss:.4f}/{vl_acc:.2f}% │ "
              f"lr={cur_lr:.2e}  "
              f"{'★' if vl_acc > best_val_acc else ''}")

        if vl_acc > best_val_acc:
            best_val_acc      = vl_acc
            best_state        = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

            # LR cai pela metade a cada lr_patience épocas sem melhora
            if epochs_no_improve % lr_patience == 0:
                new_lr = max(cur_lr * 0.5, lr_min)
                if new_lr < cur_lr:
                    for pg in optimizer.param_groups:
                        pg["lr"] = new_lr
                    print(f"    ↘  LR reduzido: {cur_lr:.2e} → {new_lr:.2e}")

            # Early stop após early_stop épocas sem melhora
            if epochs_no_improve >= early_stop:
                print(f"    🛑 Early stopping na época {epoch+1} "
                      f"({early_stop} épocas sem melhora)")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return best_val_acc, history

# ══════════════════════════════════════════════════════════════════════════════
# BUSCA PARA UM ÚNICO GRUPO
# ══════════════════════════════════════════════════════════════════════════════

def search_grupo(modelo, mod_classes):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print(f"\n{sep}")
    print(f"  🔍  GRUPO: {modelo}  │  device: {device}")
    print(sep)

    # 1. HDF5
    print(f"\n[1/4] HDF5...")
    h5_loc, num_classes, group_indices, global_to_local = \
        ensure_hdf5(modelo, mod_classes)

    # 2. Split
    print(f"\n[2/4] Split de índices...")
    train_idx, val_idx, test_idx = ensure_split(modelo, h5_loc)

    # 3. Folds
    print(f"\n[3/4] Folds...")
    fold_splits = ensure_folds(modelo, h5_loc, train_idx)

    # 4. Busca
    print(f"\n[4/4] Busca de arquitetura  "
          f"({len(ARCHITECTURES)} arquiteturas × {K_FOLDS} folds)")

    all_results, done_set = load_results(modelo)

    total = len(ARCHITECTURES) * K_FOLDS
    done  = len(done_set)
    print(f"  Total jobs: {total}  │  Concluídos: {done}"
          f"  │  Restantes: {total - done}\n")

    best_mean = max(
        (r["mean_val_acc"] for r in update_summary(
            modelo, all_results, num_classes)
         if r.get("complete")),
        default=-1.0
    )

    for arch_idx, arch_entry in enumerate(ARCHITECTURES, 1):
        label = arch_entry["label"]
        arch  = arch_entry["arch"]

        folds_done = [f for f in range(K_FOLDS) if (label, f) in done_set]
        if len(folds_done) == K_FOLDS:
            accs = [r["best_val_acc"] for r in all_results
                    if r["label"] == label]
            print(f"  ⏭  [{arch_idx}/{len(ARCHITECTURES)}] "
                  f"{label}  (média={np.mean(accs):.2f}%)")
            continue

        print(f"\n{sep}")
        print(f"  [{arch_idx}/{len(ARCHITECTURES)}]  "
              f"{modelo}  │  {label}")
        print(f"  Filtros : " +
              " → ".join(str(b["out_channels"]) for b in arch))
        restantes = [f+1 for f in range(K_FOLDS)
                     if (label, f) not in done_set]
        print(f"  Folds restantes: {restantes}")
        print(sep)

        t0_arch = time.time()

        for fold_i, (tr_pos, vl_pos) in enumerate(fold_splits):

            if (label, fold_i) in done_set:
                acc = next(r["best_val_acc"] for r in all_results
                           if r["label"] == label and r["fold"] == fold_i)
                print(f"\n  Fold {fold_i+1}/{K_FOLDS} — ⏭  "
                      f"(val_acc={acc:.2f}%)")
                continue

            print(f"\n  ── Fold {fold_i+1}/{K_FOLDS} "
                  f"(treino={len(tr_pos):,}  val={len(vl_pos):,}) ──")

            # Seed determinístico e único por (arch, fold)
            fold_seed = SEED + arch_idx * 100 + fold_i
            random.seed(fold_seed)
            np.random.seed(fold_seed)
            torch.manual_seed(fold_seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(fold_seed)

            g = torch.Generator()
            g.manual_seed(fold_seed)

            tr_abs = train_idx[tr_pos]
            vl_abs = train_idx[vl_pos]

            tr_loader = DataLoader(
                H5PyDataset(h5_loc, 'X', 'Y', 'Z', tr_abs, formats=0),
                batch_size=BATCH_SIZE, shuffle=True,
                num_workers=0, generator=g
            )
            vl_loader = DataLoader(
                H5PyDataset(h5_loc, 'X', 'Y', 'Z', vl_abs, formats=0),
                batch_size=BATCH_SIZE, shuffle=False, num_workers=0
            )

            model    = FlexCNN(num_classes=num_classes, arch=arch,
                               classifier=CLASSIFIER_HEAD, dropout=DROPOUT)
            n_params = sum(p.numel() for p in model.parameters()
                          if p.requires_grad)
            print(f"  Parâmetros: {n_params:,}")

            t0_fold = time.time()

            best_val_acc, history = train_one_fold(
                model      = model,
                tr_loader  = tr_loader,
                vl_loader  = vl_loader,
                device     = device,
                lr         = LEARNING_RATE,
                epochs     = EPOCHS_PER_FOLD,
                lr_patience= LR_PATIENCE,
                early_stop = EARLY_STOP,
                lr_min     = LR_MIN,
            )

            elapsed = time.time() - t0_fold
            print(f"\n  ✅ Fold {fold_i+1}/{K_FOLDS}  "
                  f"val_acc={best_val_acc:.2f}%  ({elapsed:.0f}s)")

            fold_result = {
                "modelo"      : modelo,
                "num_classes" : num_classes,
                "label"       : label,
                "arch_idx"    : arch_idx,
                "fold"        : fold_i,
                "best_val_acc": round(best_val_acc, 4),
                "elapsed_s"   : round(elapsed, 1),
                "n_params"    : n_params,
                "fold_seed"   : fold_seed,
                "arch"        : arch,
                "n_layers"    : len(arch),
                "filters"     : [b["out_channels"] for b in arch],
                "classifier"  : CLASSIFIER_HEAD,
                "dropout"     : DROPOUT,
                "lr"          : LEARNING_RATE,
                "lr_patience" : LR_PATIENCE,
                "early_stop"  : EARLY_STOP,
                "history"     : history,
            }

            # Salva imediatamente — não perde progresso
            save_fold_result(modelo, fold_result, all_results)
            done_set.add((label, fold_i))
            update_summary(modelo, all_results, num_classes)

            model.cpu()
            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        # Estatísticas ao completar todos os folds da arquitetura
        accs_arch = [r["best_val_acc"] for r in all_results
                     if r["label"] == label]
        if len(accs_arch) == K_FOLDS:
            print(f"\n  📊 {modelo} │ {label}")
            print(f"     Folds     : {[round(a, 2) for a in accs_arch]}")
            print(f"     Média     : {np.mean(accs_arch):.2f}%")
            print(f"     Std       : {np.std(accs_arch):.2f}%")
            print(f"     Variância : {np.var(accs_arch):.4f}")
            print(f"     Tempo     : {time.time()-t0_arch:.0f}s")
            if np.mean(accs_arch) > best_mean:
                best_mean = np.mean(accs_arch)
                print(f"  🏆 Novo melhor: {label}  "
                      f"(média={best_mean:.2f}%)")

    # Sumário final do grupo
    summary = update_summary(modelo, all_results, num_classes)
    _print_summary(modelo, summary)
    return summary


def _print_summary(modelo, summary):
    print(f"\n{sep}")
    print(f"  🏆  RANKING — {modelo}")
    print(f"{'─'*65}")
    print(f"  {'Arquitetura':<42} {'Folds':>6} "
          f"{'Média':>8} {'Std':>7} {'Var':>8}")
    print(f"{'─'*65}")
    for s in summary[:5]:
        flag   = "✅" if s["complete"] else "⏳"
        status = f"{s['folds_done']}/{K_FOLDS}"
        print(f"  {flag} {s['label'][:40]:<40} {status:>6}"
              f" {s['mean_val_acc']:>7.2f}%"
              f" {s['std_val_acc']:>6.2f}%"
              f" {s['var_val_acc']:>8.4f}")
    print(sep)

# ══════════════════════════════════════════════════════════════════════════════
# LOOP PRINCIPAL — percorre todos os grupos automaticamente
# ══════════════════════════════════════════════════════════════════════════════

def run_all():
    mod_classes = json.load(open(modulation_classes_path))

    print(f"\n{'#'*65}")
    print(f"  BUSCA AUTOMÁTICA — {len(GRUPOS_ALVO)} grupos")
    print(f"  Grupos: {GRUPOS_ALVO}")
    print(f"  LR patience: {LR_PATIENCE} épocas  │  "
          f"Early stop: {EARLY_STOP} épocas")
    print(f"{'#'*65}")

    for gi, modelo in enumerate(GRUPOS_ALVO, 1):

        # Verifica se o grupo já está completamente processado
        if grupo_completo(modelo):
            print(f"\n  ⏭  [{gi}/{len(GRUPOS_ALVO)}] {modelo} — "
                  f"já completo, pulando")

            # Mostra sumário do grupo já concluído
            sp = _summary_path(modelo)
            if os.path.exists(sp):
                with open(sp) as f:
                    summary = json.load(f)
                if summary:
                    best = next((s for s in summary if s.get("complete")),
                                summary[0])
                    print(f"     Melhor: {best['label']}"
                          f"  (média={best['mean_val_acc']:.2f}%)")
            continue

        print(f"\n{'#'*65}")
        print(f"  [{gi}/{len(GRUPOS_ALVO)}] Iniciando grupo: {modelo}")
        print(f"{'#'*65}")

        t0 = time.time()
        search_grupo(modelo, mod_classes)
        elapsed = time.time() - t0
        print(f"\n  ⏱  Grupo '{modelo}' concluído em "
              f"{elapsed/60:.1f} min")

    # Sumário global
    print(f"\n{'#'*65}")
    print(f"  ✅  BUSCA COMPLETA — todos os grupos processados")
    print(f"{'#'*65}")
    for modelo in GRUPOS_ALVO:
        sp = _summary_path(modelo)
        if os.path.exists(sp):
            with open(sp) as f:
                summary = json.load(f)
            best = next((s for s in summary if s.get("complete")),
                        None)
            if best:
                print(f"  {modelo:<6} → {best['label']:<45}"
                      f"  {best['mean_val_acc']:.2f}% "
                      f"± {best['std_val_acc']:.2f}%")


# ══════════════════════════════════════════════════════════════════════════════
# EXECUÇÃO
# ══════════════════════════════════════════════════════════════════════════════

run_all()


#################################################################
  BUSCA AUTOMÁTICA — 3 grupos
  Grupos: ['AM', 'APSK', 'QAM']
  LR patience: 5 épocas  │  Early stop: 15 épocas
#################################################################

#################################################################
  [1/3] Iniciando grupo: AM
#################################################################

═════════════════════════════════════════════════════════════════
  🔍  GRUPO: AM  │  device: cuda
═════════════════════════════════════════════════════════════════

[1/4] HDF5...
  📥 Verificando se 'AM' já existe no Drive...
  📥 HDF5 baixado do Drive → /content/AM_subset_snr_0_2_4_6_8_10_12_14_16_18_20_22_24_26_28_30.hdf5

[2/4] Split de índices...
  ✅ Split carregado do Drive  (treino=157,286  val=52,429  teste=52,429)

[3/4] Folds...
  ✅ Folds carregados do Drive: /content/drive_cache/radioml_sessions/AM/kfold_folds_k5.json

[4/4] Busca de arquitetura  (12 arquiteturas × 5 folds)
  ✅ 

    Ép   1/120 │ tr=0.7644/53.42% │ vl=0.6708/55.12% │ lr=1.00e-03  ★


    Ép   2/120 │ tr=0.6992/55.15% │ vl=0.6719/55.09% │ lr=1.00e-03  


    Ép   3/120 │ tr=0.7111/55.49% │ vl=0.6699/55.28% │ lr=1.00e-03  ★


    Ép   4/120 │ tr=0.6890/55.82% │ vl=0.6772/56.44% │ lr=1.00e-03  ★


...

    Ép  70/120 │ tr=0.5868/62.91% │ vl=0.5813/63.40% │ lr=6.25e-05  


    Ép  71/120 │ tr=0.5874/63.05% │ vl=0.5797/65.08% │ lr=6.25e-05  ★


    Ép  72/120 │ tr=0.5869/64.24% │ vl=0.5762/65.92% │ lr=6.25e-05  ★


    Ép  73/120 │ tr=0.5811/64.58% │ vl=0.5726/66.64% │ lr=6.25e-05  ★


    Ép  74/120 [tr]:  97%|████▊| 1900/1967 [00:52<00:01, 36.39it/s, loss=0.5766]